# Object Detection, Segmentation and OCR

Notebook for **Chapter 9** of the *dl* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dl/ch09.html

Tip: change a number, predict the output, then run the cell again.

## 9.1 Boxes, IoU and Non-Maximum Suppression

In [ ]:
import numpy as np
def iou(a, b):
    """boxes are (x1, y1, x2, y2) with x1<x2, y1<y2"""
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area = lambda r: (r[2] - r[0]) * (r[3] - r[1])
    return inter / (area(a) + area(b) - inter)
gt   = (10, 10, 50, 50)
pred = (30, 20, 70, 60)
print("IoU", round(iou(gt, pred), 4))
print("identical", iou(gt, gt), " disjoint", iou(gt, (60, 60, 90, 90)))
print("shifted by 5 px:", round(iou(gt, (15, 10, 55, 50)), 4), " by 10 px:", round(iou(gt, (20, 10, 60, 50)), 4))

IoU 0.2308
identical 1.0  disjoint 0.0
shifted by 5 px: 0.7778  by 10 px: 0.6


In [ ]:
def nms(boxes, scores, thr=0.5):
    order = list(np.argsort(scores)[::-1]); keep = []
    while order:
        i = order.pop(0); keep.append(int(i))
        order = [j for j in order if iou(boxes[i], boxes[j]) < thr]
    return keep
boxes = [(10,10,50,50), (12,12,52,52), (14,8,54,48), (100,100,140,140), (30,30,70,70)]
scores = np.array([0.90, 0.80, 0.70, 0.85, 0.60])
for j in (1, 2, 4): print(f"IoU(box0, box{j}) = {iou(boxes[0], boxes[j]):.3f}")
print("kept at 0.5:", nms(boxes, scores, 0.5))
print("kept at 0.9:", nms(boxes, scores, 0.9))

IoU 0.2308
identical 1.0  disjoint 0.0
shifted by 5 px: 0.7778  by 10 px: 0.6
IoU(box0, box1) = 0.822
IoU(box0, box2) = 0.747
IoU(box0, box4) = 0.143
kept at 0.5: [0, 3, 4]
kept at 0.9: [0, 3, 1, 2, 4]


## 9.2 Evaluating Detectors: Precision, Recall, AP and mAP

In [ ]:
# 6 detections for class "car", ranked by confidence; 4 real cars in the dataset
tp = np.array([1, 1, 0, 1, 0, 1])          # 1 = matched a real car with IoU >= 0.5 (not already matched)
n_gt = 4
ctp = tp.cumsum(); cfp = (1 - tp).cumsum()
prec = ctp / (ctp + cfp); rec = ctp / n_gt
print("rank      ", np.arange(1, 7))
print("precision ", prec.round(3))
print("recall    ", rec.round(3))
# AP = area under the precision envelope (all-point interpolation)
mrec = np.concatenate([[0], rec, [1]]); mpre = np.concatenate([[0], prec, [0]])
for i in range(len(mpre) - 2, -1, -1): mpre[i] = max(mpre[i], mpre[i + 1])   # make precision non-increasing
idx = np.where(mrec[1:] != mrec[:-1])[0]
ap = ((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]).sum()
print("envelope  ", mpre[1:-1].round(3))
print("AP@0.5 =", round(float(ap), 4))

IoU 0.2308
identical 1.0  disjoint 0.0
shifted by 5 px: 0.7778  by 10 px: 0.6
rank       [1 2 3 4 5 6]
precision  [1.    1.    0.667 0.75  0.6   0.667]
recall     [0.25 0.5  0.5  0.75 0.75 1.  ]
envelope   [1.    1.    0.75  0.75  0.667 0.667]
AP@0.5 = 0.8542


In [ ]:
ap_car, ap_person, ap_dog = 0.8542, 0.80, 0.55   # AP@0.5 per class (car = the 0.8542 computed above; the other two are made-up)
print("mAP@0.5 =", round((ap_car + ap_person + ap_dog) / 3, 4))
# COCO style: average AP over IoU thresholds 0.50, 0.55, ..., 0.95 (made-up values that fall as the bar rises)
ap_by_thr = np.array([0.80, 0.77, 0.73, 0.68, 0.62, 0.55, 0.46, 0.35, 0.21, 0.07])
print("thresholds", np.round(np.arange(0.5, 1.0, 0.05), 2))
print("mAP@[.5:.95] =", round(float(ap_by_thr.mean()), 3))

IoU 0.2308
identical 1.0  disjoint 0.0
shifted by 5 px: 0.7778  by 10 px: 0.6
mAP@0.5 = 0.7347
thresholds [0.5  0.55 0.6  0.65 0.7  0.75 0.8  0.85 0.9  0.95]
mAP@[.5:.95] = 0.524


## 9.3 Detector Families: R-CNN, SSD, YOLO and DETR

In [ ]:
# YOLO-style grid assignment: 448x448 image, 7x7 grid, one object box
W = H = 448; S = 7; cell = W / S
box = (100, 150, 260, 330)                         # x1, y1, x2, y2 in pixels
cx, cy = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2
bw, bh = box[2] - box[0], box[3] - box[1]
col, row = int(cx // cell), int(cy // cell)
tx, ty = cx / cell - col, cy / cell - row          # centre position inside its cell (0..1)
print("centre", (cx, cy), "-> cell (row, col) =", (row, col))
print("target: x_in_cell", round(tx, 3), "y_in_cell", round(ty, 3), "w", round(bw / W, 3), "h", round(bh / H, 3), "confidence 1")
B, C = 2, 20
print("YOLOv1 output size per image:", S * S * (B * 5 + C))

IoU 0.2308
identical 1.0  disjoint 0.0
shifted by 5 px: 0.7778  by 10 px: 0.6
centre (180.0, 240.0) -> cell (row, col) = (3, 2)
target: x_in_cell 0.812 y_in_cell 0.75 w 0.357 h 0.402 confidence 1
YOLOv1 output size per image: 1470


In [ ]:
from itertools import permutations
# DETR-style one-to-one matching: 3 predictions, 2 real objects (+ 1 "no object")
cost = np.array([[0.2, 0.9],      # cost[i][j] = how badly prediction i fits real object j
                 [0.8, 0.1],
                 [0.7, 0.6]])
best = min(permutations(range(3), 2), key=lambda p: cost[p[0], 0] + cost[p[1], 1])
total = cost[best[0], 0] + cost[best[1], 1]
print("prediction for object 0:", best[0], "| for object 1:", best[1], "| total cost", round(float(total), 2))
print("left-over prediction -> trained to say 'no object':", ({0, 1, 2} - set(best)).pop())
greedy0 = int(cost[:, 0].argmin()); print("greedy pick for object 0:", greedy0)

IoU 0.2308
identical 1.0  disjoint 0.0
shifted by 5 px: 0.7778  by 10 px: 0.6
prediction for object 0: 0 | for object 1: 1 | total cost 0.3
left-over prediction -> trained to say 'no object': 2
greedy pick for object 0: 0


In [ ]:
# Ultralytics YOLO: fine-tune a pretrained model on your labelled data (not run here)
from ultralytics import YOLO
model = YOLO("yolov8n.pt")                       # a small pretrained checkpoint
model.train(data="parts.yaml", epochs=50, imgsz=640)   # parts.yaml lists image folders and class names
results = model.predict("test.jpg", conf=0.25)   # confidence threshold

# torchvision Faster R-CNN with a pretrained backbone (not run here)
import torchvision
det = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
det.eval()   # forward(list_of_image_tensors) returns boxes, labels, scores per image

## 9.4 Segmentation: Semantic, Instance, Panoptic, U-Net and SAM

In [ ]:
# U-Net (original paper style, padded convs): size and channels at each level for a 256x256 input
h, c = 256, 1
chs = [64, 128, 256, 512, 1024]
print("encoder")
for lvl, ch in enumerate(chs):
    print(f"  level {lvl}: {h}x{h}x{ch}")
    if lvl < 4: h //= 2
print("decoder (upsample x2, concat with encoder level, 2 convs)")
for lvl in range(3, -1, -1):
    h *= 2; ch = chs[lvl]
    print(f"  level {lvl}: up to {h}x{h}, concat {ch}+{ch} = {2*ch} channels -> conv -> {ch}")
print("final 1x1 conv -> classes")
# transposed conv 2x2 stride 2 on a 2x2 input: each input value paints a 2x2 block weighted by the kernel
x = np.array([[1., 2.], [3., 4.]]); k = np.array([[1., 0.5], [0.5, 0.25]])
out = np.zeros((4, 4))
for i in range(2):
    for j in range(2): out[2*i:2*i+2, 2*j:2*j+2] += x[i, j] * k
print(out)

IoU 0.2308
identical 1.0  disjoint 0.0
shifted by 5 px: 0.7778  by 10 px: 0.6
encoder
  level 0: 256x256x64
  level 1: 128x128x128
  level 2: 64x64x256
  level 3: 32x32x512
  level 4: 16x16x1024
decoder (upsample x2, concat with encoder level, 2 convs)
  level 3: up to 32x32, concat 512+512 = 1024 channels -> conv -> 512
  level 2: up to 64x64, concat 256+256 = 512 channels -> conv -> 256
  level 1: up to 128x128, concat 128+128 = 256 channels -> conv -> 128
  level 0: up to 256x256, concat 64+64 = 128 channels -> conv -> 64
final 1x1 conv -> classes
[[1.   0.5  2.   1.  ]
 [0.5  0.25 1.   0.5 ]
 [3.   1.5  4.   2.  ]
 [1.5  0.75 2.   1.  ]]


In [ ]:
pred = np.array([[0,1,1,0],[0,1,1,0],[0,1,0,0],[0,0,0,0]])
gt   = np.array([[0,0,1,1],[0,1,1,1],[0,1,1,0],[0,0,0,0]])
inter = (pred & gt).sum(); a, b = pred.sum(), gt.sum(); union = (pred | gt).sum()
print("pred area", a, "gt area", b, "intersection", inter, "union", union)
print("IoU  ", round(inter / union, 4))
print("Dice ", round(2 * inter / (a + b), 4))
# pixel accuracy is fooled by background
print("pixel accuracy", (pred == gt).mean())

IoU 0.2308
identical 1.0  disjoint 0.0
shifted by 5 px: 0.7778  by 10 px: 0.6
pred area 5 gt area 7 intersection 4 union 8
IoU   0.5
Dice  0.6667
pixel accuracy 0.75


## 9.5 OCR and Other Vision Tasks

In [ ]:
def edit_distance(a, b):
    d = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, d[0] = d[0], i
        for j, cb in enumerate(b, 1):
            prev, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, prev + (ca != cb))
    return d[-1]
truth = "INVOICE 2024-0517"
ocr   = "INV0ICE 2O24-517"
e = edit_distance(truth, ocr)
print("edits", e, "CER", round(e / len(truth), 4))
w_t, w_o = truth.split(), ocr.split()
print("WER", edit_distance(w_t, w_o) / len(w_t))

IoU 0.2308
identical 1.0  disjoint 0.0
shifted by 5 px: 0.7778  by 10 px: 0.6
edits 3 CER 0.1765
WER 1.0


In [ ]:
import pytesseract, easyocr
from PIL import Image
text = pytesseract.image_to_string(Image.open("scan.png"), lang="eng")   # needs the Tesseract program installed
reader = easyocr.Reader(["en"])                                          # downloads models on first use
result = reader.readtext("photo.jpg")   # list of (box, text, confidence)

In [ ]:
rng = np.random.default_rng(0)
def unit(v): return v / np.linalg.norm(v)
person = unit(rng.normal(size=64))
same = [unit(person + 1.6 * unit(rng.normal(size=64))) for _ in range(300)]   # other photos of the same person
others = [unit(rng.normal(size=64)) for _ in range(300)]                      # different people
s_same = np.array([person @ v for v in same]); s_diff = np.array([person @ v for v in others])
print("same-person cosine mean", s_same.mean().round(3), " different-person mean", s_diff.mean().round(3))
for thr in (0.2, 0.3, 0.4, 0.5):
    frr = (s_same < thr).mean(); far = (s_diff >= thr).mean()
    print(f"threshold {thr}: false reject rate {frr:.3f}, false accept rate {far:.3f}")

IoU 0.2308
identical 1.0  disjoint 0.0
shifted by 5 px: 0.7778  by 10 px: 0.6
same-person cosine mean 0.517  different-person mean -0.007
threshold 0.2: false reject rate 0.000, false accept rate 0.033
threshold 0.3: false reject rate 0.007, false accept rate 0.007
threshold 0.4: false reject rate 0.087, false accept rate 0.000
threshold 0.5: false reject rate 0.410, false accept rate 0.000


## 9.6 Modern Detection and Segmentation Details: COCO mAP, Anchor-Free Decoding, ROIAlign and SAM

Section 9.2 showed AP and mAP, and Section 9.3 named the detector families. Here you compute four details that interviews and real projects hit: the full COCO score over ten IoU thresholds from actual boxes, how anchor-free heads turn raw numbers into boxes, why Mask R-CNN replaced ROI Pool with ROIAlign, and how SAM is prompted.

In [ ]:
import numpy as np
def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1]); x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2-x1)*max(0, y2-y1)
    return inter/((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter)
gt = {1: [[10,10,110,110],[200,200,260,280]], 2: [[50,50,150,120]], 3: [[30,30,90,90]]}
preds = [(1,0.90,[12,12,108,112]), (1,0.80,[205,205,262,285]), (2,0.85,[60,55,160,130]),
         (2,0.60,[50,50,150,120]), (3,0.70,[40,40,100,100]), (1,0.30,[300,300,340,340])]
preds.sort(key=lambda p: -p[1])
n_gt = sum(len(v) for v in gt.values())
def ap_at(th):
    used = {k: [False]*len(v) for k, v in gt.items()}
    tp = []
    for img, sc, box in preds:
        ious = [iou(box, g) for g in gt[img]]
        j = int(np.argmax(ious))
        if ious[j] >= th and not used[img][j]:
            used[img][j] = True; tp.append(1)
        else: tp.append(0)
    tp = np.array(tp); ctp = np.cumsum(tp); cfp = np.cumsum(1-tp)
    rec = ctp/n_gt; prec = ctp/(ctp+cfp)
    mrec = np.r_[0, rec, 1]; mpre = np.r_[0, prec, 0]
    for i in range(len(mpre)-2, -1, -1): mpre[i] = max(mpre[i], mpre[i+1])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(((mrec[idx+1]-mrec[idx])*mpre[idx+1]).sum())
print("IoU of each prediction with its best ground truth:")
for img, sc, box in preds:
    print(f"  image {img} score {sc:.2f} IoU {max(iou(box,g) for g in gt[img]):.4f}")
ths = np.arange(0.5, 0.96, 0.05)
aps = [ap_at(t) for t in ths]
for t, a in zip(ths, aps): print(f"AP@{t:.2f} = {a:.4f}")
print("AP@0.50 =", round(aps[0],4), " AP@0.75 =", round(aps[5],4), " mAP@[.5:.95] =", round(float(np.mean(aps)),4))

IoU of each prediction with its best ground truth:
  image 1 score 0.90 IoU 0.9231
  image 2 score 0.85 IoU 0.6763
  image 1 score 0.80 IoU 0.7880
  image 3 score 0.70 IoU 0.5319
  image 2 score 0.60 IoU 1.0000
  image 1 score 0.30 IoU 0.0000
AP@0.50 = 1.0000
AP@0.55 = 0.7500
AP@0.60 = 0.7500
AP@0.65 = 0.7500
AP@0.70 = 0.5667
AP@0.75 = 0.5667
AP@0.80 = 0.3500
AP@0.85 = 0.3500
AP@0.90 = 0.3500
AP@0.95 = 0.0500
AP@0.50 = 1.0  AP@0.75 = 0.5667  mAP@[.5:.95] = 0.5483


In [ ]:
import numpy as np
px, py = 200, 150
l, t, r, b = 50, 30, 70, 90
box = (px-l, py-t, px+r, py+b)
centerness = np.sqrt(min(l,r)/max(l,r) * min(t,b)/max(t,b))
print("FCOS box:", box, " centerness:", round(float(centerness),4))
stride = 8
logits = np.full(16, -9.0); logits[[5,6,7]] = np.log([0.1, 0.7, 0.2])
p = np.exp(logits); p /= p.sum()
expect = (p*np.arange(16)).sum()
print("expected distance in bins:", round(float(expect),3), " pixels:", round(float(expect*stride),2))

FCOS box: (150, 120, 270, 240)  centerness: 0.488
expected distance in bins: 6.103  pixels: 48.82


In [ ]:
import numpy as np
F = np.arange(16, dtype=float).reshape(4,4)        # feature value = 4*row + col
def bilinear(F, y, x):
    y0, x0 = int(np.floor(y)), int(np.floor(x)); y1, x1 = min(y0+1, 3), min(x0+1, 3)
    dy, dx = y-y0, x-x0
    return (F[y0,x0]*(1-dy)*(1-dx) + F[y0,x1]*(1-dy)*dx + F[y1,x0]*dy*(1-dx) + F[y1,x1]*dy*dx)
x1, y1, x2, y2 = 0.5, 0.5, 3.5, 3.5                # box in feature-map coordinates
bw, bh = (x2-x1)/2, (y2-y1)/2                       # 2x2 output, bins 1.5 wide
align = np.array([[bilinear(F, y1+bh*(i+0.5), x1+bw*(j+0.5)) for j in range(2)] for i in range(2)])
print("sample points:", y1+bh*0.5, y1+bh*1.5)
print("ROIAlign 2x2:\n", align)
qx1, qy1, qx2, qy2 = [int(np.floor(v)) for v in (x1, y1, x2, y2)]   # ROI Pool snaps corners to whole cells
sub = F[qy1:qy2+1, qx1:qx2+1]
pool = np.array([[sub[:2,:2].max(), sub[:2,2:].max()], [sub[2:,:2].max(), sub[2:,2:].max()]])
print("quantised box:", (qx1, qy1, qx2, qy2))
print("ROI Pool (max) 2x2:\n", pool)

sample points: 1.25 2.75
ROIAlign 2x2:
 [[ 6.25  7.75]
 [12.25 13.75]]
quantised box: (0, 0, 3, 3)
ROI Pool (max) 2x2:
 [[ 5.  7.]
 [13. 15.]]


## Chapter recap

## Practice set

**1.** (easy) IoU of `(0, 0, 10, 10)` and `(5, 0, 15, 10)`. **Answer:** overlap `5 × 10 = 50`, union `100 + 100 − 50 = 150`, IoU `0.333`.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dl/ch09.html